In [1]:
from urllib.parse import urlencode, quote

def build_naukri_url(keyword: str, location: str, experience: int, job_age: int = 15) -> str:
    # Format the slug portion so location comes right after "jobs-in"
    formatted_keyword = keyword.strip().lower().replace(" ", "-")
    formatted_location = location.strip().lower().replace(" ", "-")
    
    base_path = f"https://www.naukri.com/{formatted_keyword}-jobs-in-{formatted_location}"
    
    # Keep only experience and jobAge as query parameters
    params = {
        "experience": experience,
        "jobAge": job_age
    }
    
    query_string = urlencode(params, quote_via=quote)
    
    return f"{base_path}?{query_string}"

In [2]:
url = build_naukri_url("FDE", "Hyderabad", 5, 15)
url

'https://www.naukri.com/fde-jobs-in-hyderabad?experience=5&jobAge=15'

In [3]:
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def extract_naukri_jobs_headless(keyword: str, experience: int, location: str = "hyderabad secunderabad", pageNo: int = 1):
    clean_kw = keyword.strip().lower().replace(" ", "-")
    clean_loc = location.strip().lower().replace(" ", "-")
    url = f"https://www.naukri.com/{clean_kw}-jobs-in-{clean_loc}-{pageNo}?experience={experience}"
    
    # Configure Headless Chrome
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")  # Enable new headless mode
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-blink-features=AutomationControlled")
    
    # Set standard User-Agent so headless mode isn't detected
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    
    driver = webdriver.Chrome(options=options)
    jobs_list = []
    
    try:
        print(f"Navigating in Headless mode to {url}...")
        driver.get(url)
        
        # Wait up to 15 seconds for job cards to render
        wait = WebDriverWait(driver, 15)
        wait.until(
            EC.presence_of_element_located((
                By.CSS_SELECTOR, 
                "div.srp-jobtuple-wrapper, article.jobTuple, div.tuple"
            ))
        )
        time.sleep(2)
        
        cards = driver.find_elements(By.CSS_SELECTOR, "div.srp-jobtuple-wrapper, article.jobTuple, div.tuple")
        print(f"Headless browser loaded! Found and parsing {len(cards)} job cards...\n")
        
        for card in cards:
            job = {}
            
            # Title & URL
            try:
                title_elem = card.find_element(By.CSS_SELECTOR, "a.title, a.job-title")
                job["title"] = title_elem.text.strip()
                job["url"] = title_elem.get_attribute("href")
            except:
                job["title"], job["url"] = None, None
                
            # Company
            try:
                comp_elem = card.find_element(By.CSS_SELECTOR, "a.comp-name, span.comp-name, .subTitle")
                job["company"] = comp_elem.text.strip()
            except:
                job["company"] = None
                
            # Rating
            try:
                rating_elem = card.find_element(By.CSS_SELECTOR, "span.rating, .rating")
                job["rating"] = rating_elem.text.strip()
            except:
                job["rating"] = None
                
            # Experience
            try:
                exp_elem = card.find_element(By.CSS_SELECTOR, "span.exp-wrap, span.expwdth, .experience")
                job["experience"] = exp_elem.text.strip()
            except:
                job["experience"] = None
                
            # Salary
            try:
                sal_elem = card.find_element(By.CSS_SELECTOR, "span.sal-wrap, span.ni-job-tuple-icon-srp-rupee, .salary")
                job["salary"] = sal_elem.text.strip()
            except:
                job["salary"] = None
                
            # Location
            try:
                loc_elem = card.find_element(By.CSS_SELECTOR, "span.loc-wrap, span.locWdth, .location")
                job["location"] = loc_elem.text.strip()
            except:
                job["location"] = None
                
            # Skills
            try:
                tag_elems = card.find_elements(By.CSS_SELECTOR, "ul.tags-row li, div.tags-gt li, .dot-gt li")
                job["skills"] = [t.text.strip() for t in tag_elems if t.text.strip()]
            except:
                job["skills"] = []
                
            if job["title"]:
                jobs_list.append(job)
                
    except Exception as e:
        print(f"Error in headless browser: {e}")
        
    finally:
        driver.quit()
        
    return jobs_list


In [4]:
jobs = extract_naukri_jobs_headless(keyword="fde accenture deloitte", experience=2, location="hyderabad", pageNo=3)
    
# Print formatted JSON output
print(json.dumps(jobs, indent=2, ensure_ascii=False))

Navigating in Headless mode to https://www.naukri.com/fde-accenture-deloitte-jobs-in-hyderabad-3?experience=2...
Headless browser loaded! Found and parsing 20 job cards...

[
  {
    "title": "Deloitte - Consultant/Senior Consultant - HR Transformation - UKG",
    "url": "https://www.naukri.com/job-listings-deloitte-consultant-senior-consultant-hr-transformation-ukg-deloitte-shared-services-india-hyderabad-bengaluru-mumbai-all-areas-2-to-7-years-240726014342",
    "company": "Deloitte Shared Services India",
    "rating": "3.6",
    "experience": "2-7 Yrs",
    "salary": null,
    "location": "Hyderabad, Bengaluru, Mumbai (All Areas)",
    "skills": []
  },
  {
    "title": "Custom Software Engineer",
    "url": "https://www.naukri.com/job-listings-custom-software-engineer-accenture-solutions-pvt-ltd-hyderabad-2-to-5-years-051026938874",
    "company": "Accenture",
    "rating": "3.7",
    "experience": "2-5 Yrs",
    "salary": null,
    "location": "Hyderabad",
    "skills": []
  },
 

In [20]:
import json
import time, re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import urlencode, quote
from bs4 import BeautifulSoup

class NaukriSearch:
    def __init__(self):
        pass

    
    def build_naukri_url(self, keyword: str, location: str, experience: int, job_age: int = 15) -> str:
        # Format the slug portion so location comes right after "jobs-in"
        formatted_keyword = keyword.strip().lower().replace(" ", "-")
        formatted_location = location.strip().lower().replace(" ", "-")
        
        base_path = f"https://www.naukri.com/{formatted_keyword}-jobs-in-{formatted_location}"
        
        # Keep only experience and jobAge as query parameters
        params = {
            "experience": experience,
            "jobAge": job_age
        }
        
        query_string = urlencode(params, quote_via=quote)
        
        return f"{base_path}?{query_string}"
    
    def search(self, keywords: str, location: str, experience: int, pageNo: int = -1):
        clean_kw = keywords.strip().lower().replace(" ", "-")
        clean_loc = location.strip().lower().replace(" ", "-")
        url = f"https://www.naukri.com/{clean_kw}-jobs-in-{clean_loc}{f'-{pageNo}' if pageNo != -1 else ''}?experience={experience}"
        
        # Configure Headless Chrome
        options = webdriver.ChromeOptions()
        options.add_argument("--headless=new")  # Enable new headless mode
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--disable-blink-features=AutomationControlled")
        
        # Set standard User-Agent so headless mode isn't detected
        options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        
        driver = webdriver.Chrome(options=options)
        jobs_list = []
        
        try:
            print(f"Navigating in Headless mode to {url}...")
            driver.get(url)
            
            # Wait up to 15 seconds for job cards to render
            wait = WebDriverWait(driver, 15)
            wait.until(
                EC.presence_of_element_located((
                    By.CSS_SELECTOR, 
                    "div.srp-jobtuple-wrapper, article.jobTuple, div.tuple"
                ))
            )
            time.sleep(2)
            html = driver.page_source
            jobs = self.extract_jobs_from_html(html)
            return jobs
        except Exception as e:
            print("Error Can't parse jobs from html")
            print(e)
            return []
            
    
    def extract_jobs_from_html(self, html_content: str):
        soup = BeautifulSoup(html_content, "html.parser")
        
        # 1. Main card container selector
        CARD_SELECTOR = "div.srp-jobtuple-wrapper, article.jobTuple, div.tuple"
        
        # 2. Schema mapping: field_name -> (css_selector, extraction_type)
        SCHEMA = {
            "title":      ("a.title, a.job-title", "text"),
            "url":        ("a.title, a.job-title", "href"),
            "company":    ("a.comp-name, span.comp-name, .subTitle", "text"),
            "rating":     ("span.rating, .rating", "text"),
            "experience": ("span.exp-wrap, span.expwdth, .experience", "text"),
            "salary":     ("span.sal-wrap, .salary", "text"),
            "location":   ("span.loc-wrap, span.locWdth, .location", "text"),
            "skills":     ("ul.tags-row li, div.tags-gt li, .dot-gt li", "list")
        }
        jobs = []
        
        # Iterate through each job card container
        for card in soup.select(CARD_SELECTOR):
            job = {}
            for field, (selector, extract_type) in SCHEMA.items():
                elem = card.select_one(selector)
                
                if extract_type == "list":
                    job[field] = [tag.get_text(strip=True) for tag in card.select(selector) if tag.get_text(strip=True)]
                elif extract_type == "href":
                    job[field] = elem.get("href") if elem else None
                else:  # "text"
                    job[field] = elem.get_text(strip=True) if elem else None
                    
            if job["title"]:
                jobs.append(job)
        return jobs


    def get_job_description(self, job_url):
        # Configure Headless Chrome
        options = webdriver.ChromeOptions()
        options.add_argument("--headless=new")  # Enable new headless mode
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--disable-blink-features=AutomationControlled")
        
        # Set standard User-Agent so headless mode isn't detected
        options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        
        driver = webdriver.Chrome(options=options)
        jobs_list = []
        
        try:
            print(f"Navigating in Headless mode to {job_url}...")
            driver.get(job_url)
            
            # Wait up to 15 seconds for job cards to render
            wait = WebDriverWait(driver, 15)
            # wait.until(
            #     EC.presence_of_element_located((
            #         By.CSS_SELECTOR, 
            #         "div.srp-jobtuple-wrapper, article.jobTuple, div.tuple"
            #     ))
            # )
            time.sleep(2)
            html = driver.page_source
            sp = BeautifulSoup(html, 'html.parser')
            job_desc = sp.find(class_=re.compile(r"^styles_job-desc-container__[a-zA-Z0-9]")).getText()
            return job_desc
        except Exception as e:
            print("Error Can't parse jobs from html")
            print(e)
            return []

In [19]:
sc = NaukriSearch()
sc.get_job_description('https://www.naukri.com/job-listings-deloitte-consultant-senior-consultant-hr-transformation-ukg-deloitte-shared-services-india-hyderabad-bengaluru-mumbai-all-areas-2-to-7-years-240726014342')

Navigating in Headless mode to https://www.naukri.com/job-listings-deloitte-consultant-senior-consultant-hr-transformation-ukg-deloitte-shared-services-india-hyderabad-bengaluru-mumbai-all-areas-2-to-7-years-240726014342...
Job description Consultant/Senior Consultant - HR Transformation - UKGYour potential, unleashed.India's impact on the global economy has increased at an exponential rate and Deloitte presents an opportunity to unleash and realise your potential amongst cutting edge leaders, and organisations shaping the future of the region, and indeed, the world beyond.At Deloitte, your whole self to work, every day. Combine that with our drive to propel with purpose and you have the perfect playground to collaborate, innovate, grow, and make an impact that matters.The team:Todays business challenges present a new wave of HR, talent and organization priorities. Deloitte's Human Capital services leverage research, analytics, and industry insights to help design and execute critical 

In [16]:
sc = NaukriSearch()
sc.search("AI ML Engineer Deloitte", "Hyderabad", 3)

Navigating in Headless mode to https://www.naukri.com/ai-ml-engineer-deloitte-jobs-in-hyderabad?experience=3...


[{'title': 'Deloitte - Consultant/Senior Consultant - HR Transformation - UKG',
  'url': 'https://www.naukri.com/job-listings-deloitte-consultant-senior-consultant-hr-transformation-ukg-deloitte-shared-services-india-hyderabad-bengaluru-mumbai-all-areas-2-to-7-years-240726014342',
  'company': 'Deloitte Shared Services India',
  'rating': '3.6',
  'experience': '2-7 Yrs',
  'salary': None,
  'location': 'Hyderabad, Bengaluru, Mumbai (All Areas)',
  'skills': []},
 {'title': 'Sap Abap Consultant',
  'url': 'https://www.naukri.com/job-listings-sap-abap-consultant-deloitte-shared-services-india-hyderabad-3-to-8-years-051026030950',
  'company': 'Deloitte Shared Services India',
  'rating': '3.6',
  'experience': '3-8 Yrs',
  'salary': None,
  'location': 'Hyderabad',
  'skills': []},
 {'title': 'Walk-in || XSOAR / XSIAM Engineer - Analyst, Consultant, AM',
  'url': 'https://www.naukri.com/job-listings-xsoar-xsiam-engineer-analyst-consultant-am-deloitte-shared-services-india-mumbai-hyderab

In [ ]:
url = 'https://www.naukri.com/job-listings-custom-software-engineer-accenture-solutions-pvt-ltd-hyderabad-2-to-5-years-051026938874'

